In [ ]:
## Import Python libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_recall_curve, auc

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import (
    cumulative_dynamic_auc,
    concordance_index_censored,
    brier_score,
)


In [ ]:


# =====================
# Load data
# =====================

Data = pd.read_csv(
    "/home/sdw/PycharmProjects/Lily_RBD_Survival/PD_Control_ML_Fulldata.csv"
)

print(Data["Outcome"].value_counts())


predictors = [
    "dur_w_min",
    "arm_swing_amplitude_mean_w", "arm_swing_amplitude_var_w",
    "cadence_mean_w_min", "cadence_var_w",
    "jerks_mean_w", "jerks_var_w", "dur_j_min",
    "arm_swing_amplitude_mean_j", "arm_swing_amplitude_var_j",
    "cadence_mean_j_min", "cadence_var_j",
    "jerks_mean_j", "jerks_var_j",
    "Age_accele", "Gender", "Height", "BMI",
]


# =====================
# Helpers
# =====================

def make_surv(df):

    return np.array(
        [(bool(e), t)
         for e, t in zip(
            df["Outcome"],
            df["FU_duration"]
         )],
        dtype=[("event", "?"), ("time", "<f8")]
    )


def compute_auprc(risk_scores, times, test_event, test_time):

    out = []

    for i, t in enumerate(times):

        event_before = (
            (test_event == 1) &
            (test_time <= t)
        )

        if event_before.sum() < 2:
            out.append(np.nan)
            continue

        precision, recall, _ = precision_recall_curve(
            event_before.astype(int),
            risk_scores[:, i]
        )

        out.append(auc(recall, precision))

    return np.array(out)


# =====================
# Settings
# =====================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=124
)

plot_times = np.linspace(
    2,
    Data["FU_duration"].max(),
    20
)

intervals = [(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9)]


results = {
    f"{a}-{b}": {
        "cindex": [],
        "ibs": [],
        "auprc": [],
        "auroc": []
    }
    for a, b in intervals
}


# =====================
# CV loop
# =====================

for fold, (train_idx, test_idx) in enumerate(
        skf.split(Data, Data["Outcome"])
):

    print("Fold", fold + 1)

    train = Data.iloc[train_idx]
    test = Data.iloc[test_idx]

    X_train = train[predictors]
    X_test = test[predictors]

    imp = SimpleImputer(strategy="median")

    X_train = imp.fit_transform(X_train)
    X_test = imp.transform(X_test)

    y_train = make_surv(train)
    y_test = make_surv(test)

    rsf = RandomSurvivalForest(
    n_estimators=800,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="log2",
    n_jobs=-1,
    random_state=124 + fold,
)

    rsf.fit(X_train, y_train)

    surv_funcs = rsf.predict_survival_function(X_test)

    valid_times = plot_times[
        (plot_times >= y_test["time"].min()) &
        (plot_times <= y_test["time"].max() - 1e-8)
    ]

    if len(valid_times) == 0:
        continue

    surv_probs = np.vstack(
        [fn(valid_times) for fn in surv_funcs]
    )

    risk = 1 - surv_probs

    test_event = test["Outcome"].values
    test_time = test["FU_duration"].values


    # ========= AUROC =========

    auc_time = []

    for i, t in enumerate(valid_times):

        try:
            auc_t, _ = cumulative_dynamic_auc(
                y_train,
                y_test,
                risk[:, i],
                np.array([t])
            )
            auc_time.append(auc_t[0])

        except:
            auc_time.append(np.nan)


    # ========= C-index =========

    cindex_time = []

    for i in range(len(valid_times)):

        try:
            c = concordance_index_censored(
                y_test["event"],
                y_test["time"],
                risk[:, i]
            )[0]

            cindex_time.append(c)

        except:
            cindex_time.append(np.nan)


    # ========= IBS =========

    try:
        _, brier = brier_score(
            y_train,
            y_test,
            surv_probs,
            valid_times
        )
    except:
        brier = np.full(len(valid_times), np.nan)


    # ========= AUPRC =========

    auprc_time = compute_auprc(
        risk,
        valid_times,
        test_event,
        test_time
    )


    # ========= interval summary =========

    for a, b in intervals:

        mask = (
            (valid_times >= a) &
            (valid_times < b)
        )

        if mask.sum() == 0:
            continue

        key = f"{a}-{b}"

        results[key]["cindex"].append(
            np.nanmean(np.array(cindex_time)[mask])
        )

        results[key]["ibs"].append(
            np.nanmean(brier[mask])
        )

        results[key]["auprc"].append(
            np.nanmean(auprc_time[mask])
        )

        results[key]["auroc"].append(
            np.nanmean(np.array(auc_time)[mask])
        )


# =====================
# Print summary
# =====================

print("\nInterval summary")

for k, v in results.items():

    print(
        k,
        "C-index", np.nanmean(v["cindex"]),
        "IBS", np.nanmean(v["ibs"]),
        "AUPRC", np.nanmean(v["auprc"]),
        "AUROC", np.nanmean(v["auroc"]),
    )

In [ ]:
# ======================
# Bootstrap CI
# ======================

def bootstrap_ci(x, n_boot=1000, ci=0.95):

    x = np.array(x)
    x = x[~np.isnan(x)]

    if len(x) == 0:
        return np.nan, np.nan

    means = []

    for _ in range(n_boot):

        sample = np.random.choice(
            x,
            size=len(x),
            replace=True
        )

        means.append(np.mean(sample))

    low = np.percentile(means, (1-ci)/2*100)
    high = np.percentile(means, (1+ci)/2*100)

    return low, high



rows = []

for k, v in results.items():

    mean_c = np.nanmean(v["cindex"])
    sd_c = np.nanstd(v["cindex"], ddof=1)

    mean_i = np.nanmean(v["ibs"])
    sd_i = np.nanstd(v["ibs"], ddof=1)

    mean_p = np.nanmean(v["auprc"])
    sd_p = np.nanstd(v["auprc"], ddof=1)

    mean_a = np.nanmean(v["auroc"])
    sd_a = np.nanstd(v["auroc"], ddof=1)

    ci_c = bootstrap_ci(v["cindex"])
    ci_i = bootstrap_ci(v["ibs"])
    ci_p = bootstrap_ci(v["auprc"])
    ci_a = bootstrap_ci(v["auroc"])

    rows.append([
        k,
        mean_c, sd_c, ci_c[0], ci_c[1],
        mean_i, sd_i, ci_i[0], ci_i[1],
        mean_p, sd_p, ci_p[0], ci_p[1],
        mean_a, sd_a, ci_a[0], ci_a[1],
    ])


summary = pd.DataFrame(
    rows,
    columns=[
        "interval",

        "cindex_mean","cindex_sd","cindex_ci_low","cindex_ci_high",

        "ibs_mean","ibs_sd","ibs_ci_low","ibs_ci_high",

        "auprc_mean","auprc_sd","auprc_ci_low","auprc_ci_high",

        "auroc_mean","auroc_sd","auroc_ci_low","auroc_ci_high",
    ]
)

print(summary)

summary.to_csv("interval_summary.csv", index=False)


# ======================
# AUPRC curve per interval
# ======================

interval_labels = list(results.keys())

auprc_means = [
    np.nanmean(results[k]["auprc"])
    for k in interval_labels
]


# ======================
# Plot
# ======================

plt.figure(figsize=(7,5))

plt.plot(
    interval_labels,
    auprc_means,
    marker="o",
    linewidth=2
)

plt.xlabel("Time interval (years)")
plt.ylabel("Mean AUPRC")

plt.title("Time-dependent AUPRC")

plt.grid(True)

plt.tight_layout()

plt.savefig("auprc_curve.png", dpi=300)

plt.show()